In [114]:
#--Libraries
import pandas as pd
import numpy as np
import os
from unidecode import unidecode as ud
from faker import Faker
fake = Faker(['en_US', 'es_ES'])
rng = np.random.default_rng(seed=50)
import warnings
warnings.filterwarnings('ignore')

output_folder = (r"C:\Users\USER\Desktop\DA_Portfolio\viz1_callcenter\viz1_csv") #<< Always check the folder path
os.makedirs(output_folder, exist_ok=True)

#### **📅 Calendar Process Steps 📅**

##### This code creates a csv file with dates along with its dependecies like year, month,day,ect. It asks you enter the desired dates (start and end dates)

In [104]:
%%time

#--Step 1: Inputs parameters (start date, end date and index start)
start_date = '2015/01/01' #input("What is the first date? (YYYY-MM-DD) ")
end_date = '2015/12/31' #input("What is the last date? (YYYY-MM-DD) ")
calendar_index = 1 #int(input("What was the calendar index?: "))

#--Step 2: Dates parameters range list
dates = pd.date_range(start=start_date, end=end_date)
idx = list(range(calendar_index, calendar_index + len(dates)))

#--Step 3: Add months in spanish (only if needed)
es_month_initials = {
    1: 'E', 2: 'F', 3: 'M', 4: 'A', 5: 'M', 6: 'J',
    7: 'J', 8: 'A', 9: 'S', 10: 'O', 11: 'N', 12: 'D'
}

#--Step 4: DataFrame Columns
years = dates.year
months = dates.month
months_names = dates.month_name().str[:3].str.strip()
days = dates.day
days_week = dates.dayofweek + 1
days_name = dates.day_name().str[:3].str.strip()
months_ends = dates + pd.offsets.MonthEnd(0)
quarters = dates.quarter
week_start = dates.to_period('W-SAT').start_time
week_end = dates.to_period('W-SAT').end_time.date
weeks =  np.where(dates.dayofweek + 1 > 5, 'Weekend','Weekday')
week_year = dates.isocalendar().week
month_year =  dates.month_name().str[:3].str.strip() + "-" + dates.year.astype('str').str[-2:].str.strip()
ini_month = dates.month.map(es_month_initials)
max_year_val = np.where(dates.year.min() == dates.year.max() , dates.year.min(), dates.year.max())
max_month_val = np.where(dates.month.min() == dates.month.max(), dates.month.min(), dates.month.max())
max_week_val = week_start.max()

#--Step 5: Calendar Dataframe and its columns
calendar_df = pd.DataFrame({
    'idx': idx,
    'dates': dates,
    'years': years,
    'months': months,
    'months_name': months_names,
    'days': days,
    'days_week': days_week,
    'days_name': days_name,
    'months_ends': months_ends,
    'quarters': quarters,
    'week_start': week_start,
    'week_end': week_end,
    'weeks': weeks,
    'week_year': week_year,
    'month_year': month_year,
    'ini_month': ini_month,
    'year_offset': years - max_year_val,
    'month_offset': months - max_month_val,
    'week_offset': (week_start - max_week_val).days // 7
}).reset_index(drop=True)
# --Fix any incorrect column type
calendar_df = calendar_df.astype({
    'week_start': 'datetime64[ns]',
    'week_end': 'datetime64[ns]',
    'week_year': 'int8',
    'week_offset': 'int8'
})

# #--Optional Step: Getting the minimun and maximun years of dataframe to create name file
# min_year = calendar['years'].min()
# max_year = calendar['years'].max()

CPU times: total: 31.2 ms
Wall time: 33.4 ms


#### **🔎 Mode/Types Categories 🔍**
##### Here are all the categories of contacts type to be used

In [105]:
%%time

#--Step 1: Categories List
categories = ['Mobile','Profile','Messages','Account','Refunds','Gen.Info','Password','Payments','Ask/Locate Info']

#--Step 2: Create lists to hold the index and values
categories_list = [(idx,catg) for idx, catg in enumerate(categories, start=1)]

#--Step 3: Create dataframe
category_df = pd.DataFrame(categories_list,columns=['idx','category'])

CPU times: total: 0 ns
Wall time: 577 μs


#### **👩‍💻Acct Agents Status🧑‍💻**
##### This create the differente states of the agent account information

In [106]:
%%time

#--Step 1: List accounts and status
accts = ['Active','Supervisor','Vacation','Deactivated','Inactive']

#--Step 2: Create lists to hold the index and values
accts_list = [(idx,acct) for idx, acct in enumerate(accts, start=1)]

#--Step 3: Create dataframe
acct_df = pd.DataFrame(accts_list,columns=['idx','status'])

CPU times: total: 0 ns
Wall time: 655 μs


#### **🧑‍💻Agents Creator👩‍💻**
##### All agents names, waves, departments and other details are here.

In [107]:
%%time

#--Step 1: This function help to remove accents from spanish names/last name like {ñ,â,é}
def remove_accents(text):
    return ud(text)

#--Step 2: Use calendar file
calendar = calendar_df.iloc[:,[0,3,6,13]]

#--Step 3: Filter the calendar file to only get the first mondays of every quarter
    #--Mondays of every quarters
calendar = calendar[
    (calendar['days_week'] == 1) & #--Only Mondays
    (calendar['months'].isin([1,4,7,10])) & #--Quarter Months
    (calendar['week_year'] > 1)
]
    #--First mondays of every quarters
calendar = (
    calendar
    .groupby(['months'],as_index=False)['idx']
    .first()
)
    #--Use necessary columns
calendar = calendar['idx']

#--Step 4: Agents parameters
lobs = ['Chat','Email','Phone'] #--Departments
emp_code_start = 1
agents_amount = rng.integers(115,130,endpoint=True)
hire_month = calendar

#--Step 5: List of Variables from Series Frames, Probabilities and Random Choices
    #--Lobs list, probability and choice
lobs_choices = np.random.choice(lobs, size=agents_amount)
    #--Dates list, probability and choice
date_choices = sorted(np.random.choice(hire_month, size=agents_amount))

#--Step 6: Generate name list
first_name = [remove_accents(fake.unique.first_name()) for _ in range(agents_amount)]
last_name = [remove_accents(fake.unique.last_name()) for _ in range(agents_amount)]

#--Step 7: Create the employee code
idx = list(range(emp_code_start, emp_code_start + agents_amount))
initials = [first[0] + last[0] for first, last in zip(first_name, last_name)]
emp_codes_full = [str(i) + code for i, code in zip(idx, initials)]

#--Step 8: Create DataFrame to store full information
agents_df = pd.DataFrame({
    'idx': idx,
    'first_name': first_name,
    'last_name': last_name,
    'emp_code': emp_codes_full,
    'lob': lobs_choices,
    'hire_date': date_choices,
    'acct_status': 1
})

CPU times: total: 46.9 ms
Wall time: 47.3 ms


#### **🧑‍💻Absent/Attendance Creator👩‍💻**
##### This code take all the working days (Mon-Fri), use the agents file and randomly give them 1-3 days of absentism; also it takes the working days (not absent) and create a file to make attedance and production (calls,email,etc)

In [108]:
%%time

#--Step 1: Create random seed and attendance types
attendance_type = ['on time','late'] #--Attendance Types
rng_attendance = np.random.dirichlet(np.ones(len(attendance_type))) #--Random options of attendance types

#--Step 2: Use calendar and agents file
calendar = calendar_df.iloc[:,[0,2,3,6,10,11,13]]
agents = agents_df.iloc[:,[3,4,5]]

#--Step 3: Filter the calendar file to only get all mondays from calendar
    #--Parameters
workdays = calendar['days_week'] < 6
low_week_year = calendar['week_year'] != 1
max_week_year = calendar['week_year'] < calendar['week_year'].max()-1
valid_start_week = calendar['week_start'].dt.year >= calendar['years']
valid_end_week = calendar['week_end'].dt.year <= calendar['years']

calendar = calendar[
    workdays &
    low_week_year &
    max_week_year &
    valid_start_week &
    valid_end_week
][['idx','years','months']]

#--Step 4: Merge calendar and agents into one dataframe, filter agents hire_date to be less than calendar date
attendance_df = pd.merge(agents,calendar,how='cross').query('idx >= hire_date')

# --Step 5: Generate the target number of absences (1-3) for each month per agent
attendance_df["absences_n"] = attendance_df.groupby(["emp_code", "years", "months"])[
    "idx"
].transform(lambda x: rng.integers(1, 3, endpoint=True))

#--Step 6: Add random noise to a new column for shuffling
attendance_df["random_noise"] = rng.random(size=len(attendance_df))

#--Step 7: Shuffle the noise in order from 1 to N
attendance_df["random_day_rank"] = (
    attendance_df.groupby(["emp_code", "years", "months"])["random_noise"]
    .rank(method="first")
    .astype(int)
)

#--Step 8: Create the final 'absences' flag column; if the random rank of the day is <= the target absences_n for that month, they are absent (1), else (0)
attendance_df["is_absent"] = (
    attendance_df["random_day_rank"] <= attendance_df["absences_n"]
).astype(int)

#--Step 9: Create new empty column of attendance type reason
attendance_df['attendance_status'] = ''

#--Step 10: Create a mask for employees who are present
mask_absent = attendance_df['is_absent'] == 0

#--Step 11: Randomly assign attendance types reasons only to present employees
attendance_df.loc[mask_absent, 'attendance_status'] = np.random.choice(
    attendance_type,
    size = mask_absent.sum(),
    p = [0.85,0.15] #rng_attendance
)

#--Step 12: Assign Absent to employees who are absent
attendance_df.loc[~mask_absent, 'attendance_status'] = 'absent'

#--Step 13: Assign Absent to employees who are absent
attendance_df = attendance_df[['idx','emp_code','lob','attendance_status']]

CPU times: total: 219 ms
Wall time: 272 ms


##### **🏢 Production File Creator 🏢**

##### This code takes the working days (attendance) previously created from the absence file and add metric of production to every department (call,chat,etc)

In [109]:
%%time

#--Step 1: Create random seed and rating list
rating = [0,1]

#--Step 2: Use attendance file and category file
attendance = attendance_df
category = category_df

#--Step 3: Filter attendance file to only get work days (remove the absents)
attendance = attendance[attendance['attendance_status'] != 'absent']

#--Step 4: Create Dictionary to gather contact min/max value for each LOB
value_contact = {
    'Email': (56,96),
    'Chat': (20,40),
    'Phone': (20,40),
}

#--Step 5: Create column for random contact
attendance['contacts'] = attendance['lob'].apply(
    lambda x: rng.integers(
        value_contact[x][0],
        value_contact[x][1] + 1
    )
)

#--Step 6: Repeat each rows based on how many contacts it got
attendance = attendance.loc[attendance.index.repeat(attendance['contacts'])].reset_index(drop=True)

#--Step 7: List of Variables from Series Frames, Probabilities and Random Choices
    #>> categories idx as list, probability and choice
catgs = category['idx']
prob_catg = np.random.dirichlet(np.ones(len(catgs)))
type_catg = np.random.choice(catgs, size = len(attendance), p = prob_catg)
    #>> rates idx as list, probability and choice
prob_rate = np.random.dirichlet(np.ones(len(rating)))
type_rate = np.random.choice(rating, size = len(attendance), p = prob_rate)

#--Step 8: Create 2 new columns based on random options above on step 7
attendance['catg_idx'] = type_catg
attendance['rating'] = type_rate

#--Step 9: Create empty column first to save the AHT (this will work for Phone/Chat)
attendance['aht'] = pd.Series(pd.NaT, dtype='timedelta64[ns]')

#--Step 10: Create a mask for Phone and Chat
mask_lob = attendance['lob'].isin(['Phone', 'Chat'])

#--Step 11: Generate random minutes and seconds only for those rows
minutes_contact = np.random.randint(5, 20, size=mask_lob.sum())
second_contact = np.random.randint(0, 59, size=mask_lob.sum())

#--Step 12: Create timedelta variable
attendance.loc[mask_lob, 'aht'] = (
    pd.to_timedelta(minutes_contact, unit='m') +
    pd.to_timedelta(second_contact, unit='s')
)

#--Step 13: Filter by LOB
    #--Email LOB
emails_df = attendance[attendance['lob'] == 'Email'][['idx','emp_code','catg_idx','rating']]
    #--Phone LOB
calls_df = attendance[attendance['lob'] == 'Phone'][['idx','emp_code','catg_idx','aht','rating']]
calls_df['aht'] = calls_df['aht'].astype(str).str[-8:]
    #--Chat LOB
chats_df = attendance[attendance['lob'] == 'Chat'][['idx','emp_code','catg_idx','aht','rating']]
chats_df['aht'] = chats_df['aht'].astype(str).str[-8:]

CPU times: total: 10.9 s
Wall time: 11.1 s


#### **💯QA Score File💯**
##### This code create a file where it takes all the fridays (excluding the first and last two weeek of the year) of the calendar and agents idx, adds 2 quality assurance score between 50-100

In [115]:
%%time

#--Step 1: Use calendar and agents file
calendar_2 = calendar_df.iloc[:,[0,2,6,10,11,13]]
agents_2 = agents_df.iloc[:,[3,5]]

#--Step 2: Filter the calendar file to only get all fridays from calendar
    #--Parameters
fridays = calendar_2['days_week'] == 5
low_week_year = calendar_2['week_year'] != 1
max_week_year = calendar_2['week_year'] < calendar_2['week_year'].max()-1
valid_start_week = calendar_2['week_start'].dt.year >= calendar_2['years']
valid_end_week = calendar_2['week_end'].dt.year <= calendar_2['years']

calendar_2 = calendar_2[
    fridays &
    low_week_year &
    max_week_year &
    valid_start_week &
    valid_end_week
][['idx']]

#--Step 3: Merge calendar and agents into one dataframe, filter agents hire_date to be less than calendar date
qa_score = pd.merge(agents_2,calendar_2,how='cross').query('idx >= hire_date')

#--Step 4: Add min/max score values
low_score = rng.integers(50,100,size=len(qa_score),endpoint=True)
high_score = rng.integers(50,100,size=len(qa_score),endpoint=True)

#--Step 5: Create full DataFrame and add low/high scores
qa_score['score_1'] = low_score
qa_score['score_2'] = high_score

#--Step 6: Reorder columns and drop unneeded columns
qa_score = qa_score.reindex(columns=['idx','emp_code','score_1','score_2'])

CPU times: total: 31.2 ms
Wall time: 30.7 ms


#### **🗃️All files🗃️**
##### DataFrames, file names and folder path

In [ ]:
%%time

#--Step 1: Files Names
calendar_filename = 'calendar.csv' #f"calendar_{min_year}.csv" if min_year == max_year else f"calendar_{min_year}_to_{max_year}.csv"
category_filename = 'categories.csv'
account_status_filename = 'accts.csv'
agents_filename = 'agents.csv'
attendance_filename = 'attendance.csv'
emails_filename = 'wk_emails.csv'
phone_filename = 'wk_calls.csv'
chats_filename = 'wk_chats.csv'
qa_filename = 'qa_score.csv'

#--Step 2: Folder Path
file_calendar_path = os.path.join(output_folder, calendar_filename)
file_category_path = os.path.join(output_folder, category_filename)
file_account_status_path = os.path.join(output_folder, account_status_filename)
file_agents_path = os.path.join(output_folder,agents_filename)
file_attendance_path = os.path.join(output_folder,attendance_filename)
file_emails_path = os.path.join(output_folder,emails_filename)
file_phone_path = os.path.join(output_folder,phone_filename)
file_chats_path = os.path.join(output_folder,chats_filename)
file_qa_path = os.path.join(output_folder, qa_filename)

#--Step 3: Dataframe to CSV file
calendar_df.to_csv(file_calendar_path, index=False, encoding='utf-8')
category_df.to_csv(file_category_path, index=False, encoding='utf-8-sig')
acct_df.to_csv(file_account_status_path, index=False, encoding='utf-8-sig')
agents_df.to_csv(file_agents_path,index=False,encoding='utf-8')
attendance_df.to_csv(file_attendance_path,index=False,encoding='utf-8')
emails_df.to_csv(file_emails_path,index=False,encoding='utf-8')
calls_df.to_csv(file_phone_path,index=False,encoding='utf-8')
chats_df.to_csv(file_chats_path,index=False,encoding='utf-8')
qa_score.to_csv(file_qa_path, index=False, encoding='utf-8')

#--Step 4: Print results
print(f"File {calendar_filename} saved successfully with {len(calendar_df):,.0f} rows, please check your folder.")
print(f"File {category_filename} saved successfully with {len(category_df):,.0f} rows, please check your folder.")
print(f"File {account_status_filename} saved successfully with {len(acct_df):,.0f} rows, please check your folder.")
print(f"File {agents_filename} saved successfully with {len(agents_df):,.0f} rows, please check your folder.")
print(f"File {attendance_filename} saved successfully with {len(attendance_df):,.0f} rows, please check your folder.")
print(f"File {emails_filename} saved successfully with {len(emails_df):,.0f} rows, please check your folder.")
print(f"File {phone_filename} saved successfully with {len(calls_df):,.0f} rows, please check your folder.")
print(f"File {chats_filename} saved successfully with {len(chats_df):,.0f} rows, please check your folder.")
print(f"File {qa_filename} saved successfully with {len(qa_score):,.0f} rows, please check your folder.")